# 04 - Demostración de Persistencia Políglota (SQLite + MongoDB Atlas)

Este notebook presenta de forma integral y didáctica la arquitectura de base de datos del proyecto:
1. **Base Relacional (SQLite - `data/processed/deptos.db`):** Almacena la **Tabla 1** (`datos_limpios`), con tipado estricto, relaciones tabulares, índices optimizados y garantías ACID para análisis masivo de inmuebles.
2. **Base Documental No Relacional (MongoDB Atlas):** Almacena las **Tablas 2 y 3** y colecciones de experimentación (`parametros`, `metricas`, `predicciones`, `leakage`, `validacion_cruzada`, `logs`), permitiendo esquemas flexibles para hiperparámetros anidados, matrices de confusión y streaming de logs operacionales.

---

### Estructura de la Demostración:
- **Parte 1 — Consultas de lectura analítica sobre datos reales:**
  - *SQL en SQLite:* Conteos anuales, agregación por barrio (`GROUP BY`, `HAVING`) y filtros multivariable.
  - *NoSQL en MongoDB:* Búsquedas directas con punto (`db.parametros.find()`), conteo de documentos (`db.metricas.count_documents()`), pipeline de agregación (`db.metricas.aggregate()`) y análisis de eventos en `db.logs`.
- **Parte 2 — Cruce híbrido entre las dos bases:**
  - Recuperación de inferencias en MongoDB (`db.predicciones.find()`) y enlace con los atributos inmobiliarios originales de SQLite vía `id_fila`.
  - Auditoría de aciertos y desvíos cruzando probabilidades del modelo con características físicas del departamento.
- **Parte 3 — Ciclo CRUD completo en entorno de práctica aislado (`demo_crud`):**
  - Demostración de **C**reate, **R**ead, **U**pdate y **D**elete en tablas y colecciones temporales de práctica (`db.demo_crud`), garantizando la inmutabilidad de los datos de producción.
  - Limpieza final garantizada (*teardown* automático).


## 0. Configuración del entorno y conexión a los motores de persistencia

Se configuran las rutas del proyecto (`config.py`), se inicializa la conexión relacional mediante `sqlite3` y SQLAlchemy, y se establece la conexión directa `db` hacia **MongoDB Atlas**.


In [30]:
import sys; sys.path.append("..")
%load_ext autoreload
%autoreload 2

import sqlite3
import pandas as pd
from sqlalchemy import create_engine
from pymongo import MongoClient

import config
from src.db_sql import load_clean_data
from src.db_mongo import get_db, load_run
from src.logger_setup import get_logger

logger = get_logger("demo_bases")

# 1. Conexión Relacional a SQLite local
con_sql = sqlite3.connect(config.SQLITE_PATH)
engine_sql = create_engine(f"sqlite:///{config.SQLITE_PATH}")
print(f" Conexión establecida a SQLite: {config.SQLITE_PATH.name}")

# 2. Conexión No Relacional a MongoDB Atlas (acceso directo mediante variable 'db')
try:
    db = get_db()
    print(f" Conexión establecida a MongoDB Atlas: Base '{config.MONGO_DB}'")
    print(f"   Colecciones disponibles: {db.list_collection_names()}")
except Exception as e:
    db = None
    print(f" Advertencia: No se pudo conectar a MongoDB Atlas ({e}). Verifique conexión a internet o .env.")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
 Conexión establecida a SQLite: deptos.db
 Conexión establecida a MongoDB Atlas: Base 'taller_integrador'
   Colecciones disponibles: ['leakage', 'predicciones', 'validacion_cruzada', 'logs', 'metricas', 'parametros']


## 1. Consultas de lectura analítica sobre datos reales

---
### 1.1 Consultas SQL en la Base Relacional (SQLite - `datos_limpios`)

La tabla `datos_limpios` contiene más de 100.000 registros tabulares estandarizados. Ejecutamos tres consultas analíticas que ponen a prueba agrupaciones, filtros y funciones de agregación.


#### Consulta 1.1: Evolución temporal de publicaciones y precios por año
Agrupamos por `AÑO` calculando el volumen total de avisos, el precio por metro cuadrado promedio, la superficie promedio y el ticket promedio en dólares.


In [31]:
query_anios = """
SELECT 
    AÑO,
    COUNT(*) AS total_departamentos,
    ROUND(AVG(PRECIO_M2), 2) AS precio_prom_m2,
    ROUND(AVG(M2), 1) AS superficie_prom_m2,
    ROUND(AVG(DOLARES), 0) AS valor_prom_usd
FROM datos_limpios
GROUP BY AÑO
ORDER BY AÑO ASC;
"""

df_anios = pd.read_sql_query(query_anios, con_sql)
display(df_anios.head(10))
print(f"Total de años registrados en SQLite: {len(df_anios)}")


,AÑO,total_departamentos,precio_prom_m2,superficie_prom_m2,valor_prom_usd
0,2001,12798,890.59,83.4,73291.0
1,2002,7301,506.23,82.4,41339.0
2,2003,13393,602.03,79.0,46600.0
3,2004,4805,812.97,73.1,60441.0
4,2005,3502,915.19,66.7,61118.0
5,2006,3441,1115.79,76.0,87715.0
6,2007,7888,1300.43,85.3,112221.0
7,2008,7327,1591.31,80.5,127614.0
8,2009,23358,1693.42,78.6,137187.0
9,2010,18447,1782.23,78.9,141229.0


Total de años registrados en SQLite: 16


#### Consulta 1.2: Métricas inmobiliarias por barrio con filtro `HAVING`
Agrupamos por `BARRIO`, calculando dispersión de precios (`MIN`, `MAX`, `AVG`) y cantidad de ambientes promedio, filtrando únicamente barrios representativos con más de 1.000 avisos (`HAVING COUNT(*) > 1000`).


In [32]:
query_barrios = """
SELECT 
    BARRIO,
    COUNT(*) AS cantidad_avisos,
    ROUND(AVG(PRECIO_M2), 2) AS precio_m2_promedio,
    ROUND(MIN(PRECIO_M2), 2) AS precio_m2_min,
    ROUND(MAX(PRECIO_M2), 2) AS precio_m2_max,
    ROUND(AVG(AMBIENTES), 2) AS ambientes_promedio
FROM datos_limpios
GROUP BY BARRIO
HAVING COUNT(*) > 1000
ORDER BY precio_m2_promedio DESC
LIMIT 12;
"""

df_barrios = pd.read_sql_query(query_barrios, con_sql)
display(df_barrios)


,BARRIO,cantidad_avisos,precio_m2_promedio,precio_m2_min,precio_m2_max,ambientes_promedio
0,PUERTO MADERO,1081,4408.16,748.50,9259.26,2.96
1,RETIRO,3398,2283.01,320.86,9250.00,3.31
2,PALERMO,21570,2140.18,106.68,10000.00,2.90
3,RECOLETA,15100,2050.07,106.38,9523.81,3.07
4,NUÑEZ,4723,1964.18,249.00,6593.41,2.72
5,BELGRANO,12746,1943.20,145.15,8661.42,3.09
6,SAAVEDRA,2510,1753.46,129.17,9000.00,2.46
7,COLEGIALES,4027,1738.35,105.00,4587.50,2.78
8,COGHLAN,1261,1712.40,264.29,4524.86,2.53
9,CHACARITA,1951,1704.25,129.39,9977.78,2.39


#### Consulta 1.3: Filtro relacional multivariable (Comodidades Premium y Rango de Superficie)
Consultamos avisos específicos que cuentan simultáneamente con cochera, baulera y terraza (`COCHERA=1 AND BAULERA=1 AND TERRAZA=1`), en un rango intermedio de superficie (60 a 100 m²), ubicados en barrios de alta demanda.


In [33]:
query_filtros = """
SELECT 
    AÑO,
    BARRIO,
    AMBIENTES,
    M2,
    ROUND(DOLARES, 0) AS DOLARES,
    ROUND(PRECIO_M2, 2) AS PRECIO_M2,
    COCHERA,
    BAULERA,
    TERRAZA,
    SEGMENTO
FROM datos_limpios
WHERE COCHERA = 1.0 
  AND BAULERA = 1.0 
  AND TERRAZA = 1.0
  AND M2 BETWEEN 60 AND 100
  AND BARRIO IN ('PALERMO', 'BELGRANO', 'CABALLITO', 'RECOLETA')
ORDER BY PRECIO_M2 DESC
LIMIT 10;
"""

df_premium = pd.read_sql_query(query_filtros, con_sql)
display(df_premium)
print(f"Muestra de departamentos premium recuperada: {len(df_premium)} filas.")


,AÑO,BARRIO,AMBIENTES,M2,DOLARES,PRECIO_M2,COCHERA,BAULERA,TERRAZA,SEGMENTO
0,2014,PALERMO,3.0,97.0,450000.0,4639.18,1.0,1.0,1.0,alto
1,2014,PALERMO,2.0,75.0,320000.0,4266.67,1.0,1.0,1.0,alto
2,2014,PALERMO,3.0,85.0,345000.0,4058.82,1.0,1.0,1.0,alto
3,2014,PALERMO,3.0,72.0,277000.0,3847.22,1.0,1.0,1.0,alto
4,2014,PALERMO,4.0,63.0,242000.0,3841.27,1.0,1.0,1.0,alto
5,2014,PALERMO,3.0,81.0,270000.0,3333.33,1.0,1.0,1.0,alto
6,2014,PALERMO,4.0,93.0,310000.0,3333.33,1.0,1.0,1.0,alto
7,2014,PALERMO,3.0,73.0,240000.0,3287.67,1.0,1.0,1.0,alto
8,2014,CABALLITO,3.0,70.0,230000.0,3285.71,1.0,1.0,1.0,alto
9,2014,PALERMO,3.0,60.0,160000.0,2666.67,1.0,1.0,1.0,alto


Muestra de departamentos premium recuperada: 10 filas.


---
### 1.2 Consultas NoSQL directas en MongoDB Atlas

En MongoDB Atlas se persisten artefactos jerárquicos y métricas de Machine Learning estructuradas en colecciones BSON. Ejecutamos consultas de lectura directa con punto (`db.<coleccion>.<operacion>()`), idénticas a los filtros de **MongoDB Compass** y la consola **`mongosh`**.


#### Consulta 1.4: Conteo directo de documentos con `count_documents` por colección y escenario


In [34]:
if db is not None:
    # Consultas directas a cada colección
    resumen_cols = [
        {
            "Colección": "parametros",
            "Total Documentos": db.parametros.count_documents({}),
            "Escenario Aleatorio": db.parametros.count_documents({"escenario": "aleatorio"}),
            "Escenario Temporal": db.parametros.count_documents({"escenario": "temporal"}),
            "Escenario Común": db.parametros.count_documents({"escenario": "comun"})
        },
        {
            "Colección": "metricas",
            "Total Documentos": db.metricas.count_documents({}),
            "Escenario Aleatorio": db.metricas.count_documents({"escenario": "aleatorio"}),
            "Escenario Temporal": db.metricas.count_documents({"escenario": "temporal"}),
            "Escenario Común": "-"
        },
        {
            "Colección": "predicciones",
            "Total Documentos": db.predicciones.count_documents({}),
            "Escenario Aleatorio": db.predicciones.count_documents({"escenario": "aleatorio"}),
            "Escenario Temporal": db.predicciones.count_documents({"escenario": "temporal"}),
            "Escenario Común": "-"
        },
        {
            "Colección": "logs",
            "Total Documentos": db.logs.count_documents({}),
            "Escenario Aleatorio": "-",
            "Escenario Temporal": "-",
            "Escenario Común": "-"
        }
    ]
    display(pd.DataFrame(resumen_cols))
else:
    print("Omitido: Sin conexión a Atlas.")


,Colección,Total Documentos,Escenario Aleatorio,Escenario Temporal,Escenario Común
0,parametros,4,0,0,4
1,metricas,8,4,4,-
2,predicciones,252360,139296,113064,-
3,logs,374,-,-,-


#### Consulta 1.5: Búsqueda directa de hiperparámetros con `db.parametros.find()`
Búsqueda idéntica al filtro `{modelo: "random_forest"}` de MongoDB Compass / Atlas:


In [35]:
if db is not None:
    # Consulta directa a db.parametros identica a MongoDB Compass:
    print("Consulta: db.parametros.find({\"modelo\": \"random_forest\"})")
    print()
    for p in db.parametros.find({"modelo": "random_forest"}):
        print(f"Modelo: {p['modelo'].upper()} | CV Score: {p['cv_score']:.4f}")
        print(f"  Parametros: {p['mejores_parametros']}")
        print()
    print("-" * 70)
    print("Todos los modelos del escenario comun ordenados por score:")
    for p in db.parametros.find({"escenario": "comun"}).sort("cv_score", -1):
        print(f"  - {p['modelo'].upper():15s} | F1: {p['cv_score']:.4f} | Busqueda: {p['segundos_busqueda']:.1f}s")
else:
    print("Omitido: Sin conexion a Atlas.")


Consulta: db.parametros.find({"modelo": "random_forest"})

Modelo: RANDOM_FOREST | CV Score: 0.6440
  Parametros: {'clf__n_estimators': 200, 'clf__min_samples_leaf': 5, 'clf__max_features': 0.5, 'clf__max_depth': 20}

----------------------------------------------------------------------
Todos los modelos del escenario comun ordenados por score:
  - XGBOOST         | F1: 0.6502 | Busqueda: 7.8s
  - RANDOM_FOREST   | F1: 0.6440 | Busqueda: 113.3s
  - LOGREG          | F1: 0.6114 | Busqueda: 4.2s
  - DUMMY           | F1: 0.3338 | Busqueda: 0.0s


#### Consulta 1.6: Pipeline de Agregación (`aggregate`) directo sobre `db.metricas`
Consolidamos el rendimiento promedio de exactitud y F1-Macro agrupado por algoritmo y escenario utilizando el Aggregation Framework (`$match`, `$group`, `$sort`):


In [36]:
if db is not None:
    pipeline_metricas = [
        {"$match": {"escenario": {"$in": ["aleatorio", "temporal"]}}},
        {"$group": {
            "_id": {"modelo": "$model", "escenario": "$escenario"},
            "accuracy_promedio": {"$avg": "$accuracy"},
            "f1_macro_promedio": {"$avg": "$f1_macro"},
            "registros": {"$sum": 1}
        }},
        {"$sort": {"_id.escenario": 1, "f1_macro_promedio": -1}}
    ]
    
    # Ejecución directa sobre db.metricas
    resultado_agg = list(db.metricas.aggregate(pipeline_metricas))
    
    filas_agg = [{
        "Escenario": r["_id"]["escenario"],
        "Modelo": r["_id"]["modelo"],
        "Accuracy Promedio": round(r["accuracy_promedio"], 4),
        "F1-Macro Promedio": round(r["f1_macro_promedio"], 4),
        "Documentos": r["registros"]
    } for r in resultado_agg]
    
    print("Rendimiento Multimodelo consolidado vía db.metricas.aggregate():")
    display(pd.DataFrame(filas_agg))
else:
    print("Omitido: Sin conexión a Atlas.")


Rendimiento Multimodelo consolidado vía db.metricas.aggregate():


,Escenario,Modelo,Accuracy Promedio,F1-Macro Promedio,Documentos
0,aleatorio,random_forest,0.6783,0.6778,1
1,aleatorio,xgboost,0.6693,0.6680,1
2,aleatorio,logreg,0.6191,0.6153,1
3,aleatorio,dummy,0.3325,0.3325,1
4,temporal,random_forest,0.6403,0.6405,1
5,temporal,xgboost,0.6368,0.6353,1
6,temporal,logreg,0.6078,0.6037,1
7,temporal,dummy,0.3334,0.3334,1


#### Consulta 1.7: Trazabilidad y auditoría directa sobre `db.logs`
Agrupamos los eventos de ejecución por nivel de severidad y recuperamos los registros operacionales más recientes directamente desde `db.logs`:


In [37]:
if db is not None and "logs" in db.list_collection_names():
    # 1. Agregacion para cuantificar severidades
    pipeline_logs = [
        {"$group": {
            "_id": "$level",
            "cantidad_eventos": {"$sum": 1},
            "ultimo_evento": {"$max": "$timestamp"}
        }},
        {"$sort": {"cantidad_eventos": -1}}
    ]
    res_logs = list(db.logs.aggregate(pipeline_logs))
    print("Distribucion de eventos por severidad en db.logs:")
    display(pd.DataFrame(res_logs).rename(columns={"_id": "Nivel"}))
    
    # 2. Busqueda de los ultimos 5 eventos registrados
    print("Ultimos 5 registros en db.logs:")
    ultimos_logs = list(db.logs.find({}, {"_id": 0, "timestamp": 1, "level": 1, "module": 1, "message": 1})
                                .sort("timestamp", -1)
                                .limit(5))
    display(pd.DataFrame(ultimos_logs))
else:
    print("Omitido: Coleccion de logs no disponible o sin conexion.")


Distribucion de eventos por severidad en db.logs:


,Nivel,cantidad_eventos,ultimo_evento
0,INFO,299,2026-11-15 23:11:46.613
1,WARNING,75,2026-11-15 22:23:09.738


Ultimos 5 registros en db.logs:


,timestamp,level,module,message
0,2026-11-15 23:11:46.613,INFO,src.visualization,Figura guardada: leakage.png
1,2026-11-15 23:11:46.521,INFO,src.db_mongo,Mongo 'leakage' [leakage]: 5 documentos leídos...
2,2026-11-15 23:11:37.855,INFO,src.visualization,Figura guardada: importancia_variables.png
3,2026-11-15 23:11:34.040,INFO,src.features,Target en test:\nSEGMENTO\nalto 0.333\nmed...
4,2026-11-15 23:11:34.039,INFO,src.features,Target en train:\nSEGMENTO\nalto 0.333\nme...


---
## 2. Cruce híbrido entre las dos bases de datos (Complementariedad Políglota)

### Arquitectura de Enlace:
- **MongoDB (`db.predicciones`):** Almacena la inferencia granular conteniendo el `id_fila`, las clases predichas y el vector de probabilidades estimado por cada modelo (`proba_bajo`, `proba_medio`, `proba_alto`).
- **SQLite (`datos_limpios`):** Almacena la totalidad de las características físicas y económicas del departamento (`BARRIO`, `M2`, `AMBIENTES`, `DOLARES`, `PRECIO_M2`, `ANTIGUEDAD`).
- **El enlace:** El campo `id_fila` generado durante el split de evaluación referencia de manera biunívoca la posición `(rowid - 1)` de la tabla relacional.

A continuación, extraemos inferencias con `db.predicciones.find()` y enriquecemos las predicciones con los atributos reales de SQLite para auditar aciertos y desvíos.


In [38]:
if db is not None and "predicciones" in db.list_collection_names():
    # 1. Consulta directa sobre db.predicciones: traemos 10 predicciones del modelo 'xgboost'
    filtro_pred = {"escenario": "aleatorio", "modelo": "xgboost"}
    
    preds_cursor = db.predicciones.find(
        filtro_pred,
        {"_id": 0, "id_fila": 1, "modelo": 1, "y_real": 1, "y_pred": 1, 
         "proba_bajo": 1, "proba_medio": 1, "proba_alto": 1, "acierto": 1}
    ).limit(10)
    
    df_preds_mongo = pd.DataFrame(list(preds_cursor))
    print(f"Predicciones extraídas de db.predicciones: {len(df_preds_mongo)} registros.")
    display(df_preds_mongo.head(5))
else:
    # Fallback pedagógico si no hay conexión a Mongo
    print("Generando muestra de predicciones de prueba para demostrar el cruce...")
    df_preds_mongo = pd.DataFrame({
        "id_fila": [10, 45, 120, 305, 512, 1024, 2048, 4096],
        "modelo": ["xgboost"] * 8,
        "y_real": ["medio", "alto", "bajo", "medio", "alto", "bajo", "alto", "medio"],
        "y_pred": ["medio", "alto", "medio", "medio", "medio", "bajo", "alto", "alto"],
        "proba_bajo": [0.10, 0.05, 0.45, 0.15, 0.20, 0.75, 0.05, 0.10],
        "proba_medio": [0.80, 0.15, 0.48, 0.70, 0.42, 0.20, 0.15, 0.40],
        "proba_alto": [0.10, 0.80, 0.07, 0.15, 0.38, 0.05, 0.80, 0.50],
        "acierto": [True, True, False, True, False, True, True, False]
    })


Predicciones extraídas de db.predicciones: 10 registros.


,id_fila,modelo,y_real,y_pred,proba_alto,proba_bajo,proba_medio,acierto
0,87209,xgboost,medio,alto,0.6677,0.0638,0.2684,False
1,3846,xgboost,bajo,bajo,0.0206,0.7984,0.1809,True
2,169562,xgboost,bajo,medio,0.3995,0.1133,0.4872,False
3,147970,xgboost,medio,bajo,0.1075,0.4685,0.4240,False
4,5098,xgboost,medio,bajo,0.1436,0.4633,0.3931,False


#### Consulta parametrizada en SQLite a partir de los `id_fila` obtenidos de MongoDB
Con la lista de identificadores `id_fila` recuperada de MongoDB, ejecutamos una consulta SQL con cláusula `IN (...)` en SQLite para extraer los datos físicos y económicos de cada departamento.


In [39]:
ids_muestra = tuple(int(x) for x in df_preds_mongo["id_fila"].tolist())
placeholders = ",".join(["?"] * len(ids_muestra))

query_cruce = f"""
SELECT 
    (rowid - 1) AS id_fila,
    AÑO,
    BARRIO,
    AMBIENTES,
    M2,
    ROUND(DOLARES, 0) AS DOLARES,
    ROUND(PRECIO_M2, 2) AS PRECIO_M2,
    ANTIGUEDAD,
    SEGMENTO AS segmento_real_db
FROM datos_limpios
WHERE (rowid - 1) IN ({placeholders});
"""

# Ejecución parametrizada segura en SQLite
df_inmuebles_sql = pd.read_sql_query(query_cruce, con_sql, params=ids_muestra)
print(f"Inmuebles recuperados de SQLite: {len(df_inmuebles_sql)} registros.")
display(df_inmuebles_sql.head(5))


Inmuebles recuperados de SQLite: 10 registros.


,id_fila,AÑO,BARRIO,AMBIENTES,M2,DOLARES,PRECIO_M2,ANTIGUEDAD,segmento_real_db
0,3846,2001,MATADEROS,NaN,52.0,39000.0,750.00,NaN,bajo
1,5098,2001,CHACARITA,NaN,86.0,72500.0,843.02,NaN,medio
2,7497,2001,SAN CRISTOBAL,NaN,130.0,89500.0,688.46,NaN,bajo
3,12329,2001,SAAVEDRA,NaN,36.0,45000.0,1250.00,0.0,alto
4,75714,2009,VILLA URQUIZA,3.0,71.0,98000.0,1380.28,25.0,bajo


#### Fusión analítica: Auditoría integrada de inferencia y atributos físicos
Unimos ambos DataFrames mediante `id_fila`. Esto permite analizar por qué el modelo acertó o se equivocó evaluando el precio por metro cuadrado real, el barrio y la superficie del inmueble.


In [40]:
# Merge en memoria demostrando la complementariedad políglota
df_cruce_completo = df_preds_mongo.merge(df_inmuebles_sql, on="id_fila")

# Ordenamos visualmente las columnas para auditar el desempeño
cols_reporte = [
    "id_fila", "BARRIO", "M2", "AMBIENTES", "PRECIO_M2", 
    "y_real", "y_pred", "acierto", "proba_bajo", "proba_medio", "proba_alto"
]
display(df_cruce_completo[cols_reporte])

# Resumen de auditoría
aciertos_n = df_cruce_completo['acierto'].sum()
total_n = len(df_cruce_completo)
print(f"Tasa de acierto en la muestra cruzada: {aciertos_n}/{total_n} ({aciertos_n/total_n*100:.1f}%)")


,id_fila,BARRIO,M2,AMBIENTES,PRECIO_M2,y_real,y_pred,acierto,proba_bajo,proba_medio,proba_alto
0,87209,PALERMO,48.0,2.0,1916.67,medio,alto,False,0.0638,0.2684,0.6677
1,3846,MATADEROS,52.0,NaN,750.00,bajo,bajo,True,0.7984,0.1809,0.0206
2,169562,ALMAGRO,55.0,3.0,1636.36,bajo,medio,False,0.1133,0.4872,0.3995
3,147970,VILLA URQUIZA,60.0,3.0,2316.67,medio,bajo,False,0.4685,0.4240,0.1075
4,5098,CHACARITA,86.0,NaN,843.02,medio,bajo,False,0.4633,0.3931,0.1436
5,173236,BARRACAS,50.0,2.0,940.00,bajo,bajo,True,0.9056,0.0800,0.0144
6,75714,VILLA URQUIZA,71.0,3.0,1380.28,bajo,medio,False,0.2301,0.6061,0.1637
7,7497,SAN CRISTOBAL,130.0,NaN,688.46,bajo,bajo,True,0.8563,0.0886,0.0550
8,126750,VILLA ORTUZAR,42.0,2.0,2897.07,alto,medio,False,0.1051,0.5793,0.3155
9,12329,SAAVEDRA,36.0,NaN,1250.00,alto,alto,True,0.0331,0.3217,0.6452


Tasa de acierto en la muestra cruzada: 4/10 (40.0%)


---
## 3. Ciclo CRUD completo en entorno de práctica aislado (`demo_crud`)

### Principio de Seguridad e Inmutabilidad de Datos Productivos:
Para la defensa oral y ejecución en vivo:
- **PROHIBIDO:** Modificar o escribir sobre la tabla `datos_limpios` de SQLite o sobre las colecciones productivas de MongoDB (`metricas`, `predicciones`, etc.).
- **SOLUCIÓN:** Se implementa un ciclo **CRUD completo** (**C**reate, **R**ead, **U**pdate, **D**elete) sobre una tabla y colección efímeras denominadas `demo_crud`.
- Al finalizar la demostración, el notebook ejecuta un procedimiento de limpieza (*teardown*) que elimina cualquier rastro residual, garantizando total inocuidad.


### 3.1 CRUD en Base Relacional SQLite (`demo_crud`)

---
#### Paso C (Create): Creación de tabla y altas parametrizadas


In [53]:
# 1. CREATE TABLE
con_sql = sqlite3.connect(config.SQLITE_PATH)
engine_sql = create_engine(f"sqlite:///{config.SQLITE_PATH}")
con_sql.execute("""
CREATE TABLE IF NOT EXISTS demo_crud (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    codigo_aviso TEXT UNIQUE NOT NULL,
    barrio TEXT NOT NULL,
    m2 REAL NOT NULL,
    precio_usd REAL NOT NULL,
    activo INTEGER DEFAULT 1
);
""")
con_sql.commit()
print(" Tabla 'demo_crud' creada exitosamente en SQLite.")

# 2. INSERT (Altas de prueba individuales y en lote)
datos_insert = [
    ('DEMO-001', 'PALERMO', 55.0, 145000.0, 1),
    ('DEMO-002', 'CABALLITO', 42.0, 98000.0, 1),
    ('DEMO-003', 'BELGRANO', 80.0, 210000.0, 1),
    ('DEMO-004', 'ALMAGRO', 35.0, 72000.0, 0),
]
con_sql.executemany("""
INSERT INTO demo_crud (codigo_aviso, barrio, m2, precio_usd, activo)
VALUES (?, ?, ?, ?, ?);
""", datos_insert)
con_sql.commit()
print(f" C (Create): Se insertaron {len(datos_insert)} registros de prueba en SQLite.")


 Tabla 'demo_crud' creada exitosamente en SQLite.
 C (Create): Se insertaron 4 registros de prueba en SQLite.


#### Paso R (Read): Lectura y filtrado de registros creados


In [54]:
# R (Read)
df_crud_r1 = pd.read_sql_query("""
SELECT id, codigo_aviso, barrio, m2, precio_usd, activo 
FROM demo_crud 
WHERE activo = 1;
""", con_sql)

print(" R (Read): Registros activos en 'demo_crud':")
display(df_crud_r1)


 R (Read): Registros activos en 'demo_crud':


,id,codigo_aviso,barrio,m2,precio_usd,activo
0,1,DEMO-001,PALERMO,55.0,145000.0,1
1,2,DEMO-002,CABALLITO,42.0,98000.0,1
2,3,DEMO-003,BELGRANO,80.0,210000.0,1


#### Paso U (Update): Actualización de valores y verificación inmediata


In [55]:
# U (Update): Actualizamos el precio del aviso 'DEMO-001' y ampliamos m2
con_sql.execute("""
UPDATE demo_crud
SET precio_usd = 152000.0, m2 = 58.0
WHERE codigo_aviso = 'DEMO-001';
""")
con_sql.commit()
print(" U (Update): Registro 'DEMO-001' modificado exitosamente.")

# Verificación de la modificación
df_check_update = pd.read_sql_query("""
SELECT * FROM demo_crud WHERE codigo_aviso = 'DEMO-001';
""", con_sql)
display(df_check_update)


 U (Update): Registro 'DEMO-001' modificado exitosamente.


,id,codigo_aviso,barrio,m2,precio_usd,activo
0,1,DEMO-001,PALERMO,58.0,152000.0,1


#### Paso D (Delete): Baja física de registros


In [56]:
# D (Delete): Eliminamos el aviso inactivo 'DEMO-004'
con_sql.execute("""
DELETE FROM demo_crud 
WHERE codigo_aviso = 'DEMO-004';
""")
con_sql.commit()
print(" D (Delete): Registro 'DEMO-004' eliminado exitosamente.")

# Verificación de cantidad de registros remanentes
cant_sql = con_sql.execute("SELECT COUNT(*) FROM demo_crud").fetchone()[0]
print(f"Registros restantes en SQLite demo_crud: {cant_sql}")


 D (Delete): Registro 'DEMO-004' eliminado exitosamente.
Registros restantes en SQLite demo_crud: 3


---
### 3.2 CRUD directo en Base No Relacional MongoDB Atlas (`db.demo_crud`)

---
#### Paso C (Create): Inserción documental con `db.demo_crud.insert_one` y `insert_many`


In [57]:
if db is not None:
    # Limpieza preventiva por si quedó alguna corrida previa interrumpida
    db.demo_crud.drop()
    
    # 1. Inserción individual (insert_one)
    doc_1 = {
        "codigo": "EXP-001",
        "modelo": "random_forest",
        "parametros": {"n_estimators": 150, "max_depth": 15},
        "score_f1": 0.742,
        "activo": True
    }
    res_one = db.demo_crud.insert_one(doc_1)
    print(f" C (Create - insert_one): Insertado doc con ID: {res_one.inserted_id}")
    
    # 2. Inserción masiva (insert_many)
    docs_many = [
        {"codigo": "EXP-002", "modelo": "xgboost", "parametros": {"learning_rate": 0.1, "max_depth": 5}, "score_f1": 0.768, "activo": True},
        {"codigo": "EXP-003", "modelo": "logreg", "parametros": {"C": 1.0}, "score_f1": 0.691, "activo": False},
        {"codigo": "EXP-004", "modelo": "neural_net", "parametros": {"hidden_layers": [64, 32]}, "score_f1": 0.715, "activo": True}
    ]
    res_many = db.demo_crud.insert_many(docs_many)
    print(f" C (Create - insert_many): Se insertaron {len(res_many.inserted_ids)} documentos.")
else:
    print("Omitido: Sin conexión a Atlas.")


 C (Create - insert_one): Insertado doc con ID: 6afa41e0fea4e81d29411ea9
 C (Create - insert_many): Se insertaron 3 documentos.


#### Paso R (Read): Búsquedas estructuradas con `db.demo_crud.find()` y proyecciones BSON


In [58]:
if db is not None:
    # R (Read): Búsqueda de experimentos con score_f1 >= 0.70 y activos
    cursor_r = db.demo_crud.find(
        {"score_f1": {"$gte": 0.70}, "activo": True},
        {"_id": 0, "codigo": 1, "modelo": 1, "score_f1": 1, "parametros": 1}
    ).sort("score_f1", -1)
    
    df_mongo_r = pd.DataFrame(list(cursor_r))
    print(" R (Read): Experimentos destacados en db.demo_crud:")
    display(df_mongo_r)
else:
    print("Omitido: Sin conexión a Atlas.")


 R (Read): Experimentos destacados en db.demo_crud:


,codigo,modelo,parametros,score_f1
0,EXP-002,xgboost,"{'learning_rate': 0.1, 'max_depth': 5}",0.768
1,EXP-001,random_forest,"{'n_estimators': 150, 'max_depth': 15}",0.742
2,EXP-004,neural_net,"{'hidden_layers': [64, 32]}",0.715


#### Paso U (Update): Actualización atómica con `db.demo_crud.update_one()` y operadores `$set` / `$inc`


In [59]:
if db is not None:
    # U (Update): Modificamos el score_f1 del experimento 'EXP-001' y agregamos un contador de iteraciones
    db.demo_crud.update_one(
        {"codigo": "EXP-001"},
        {
            "$set": {"score_f1": 0.755, "revisado": True},
            "$inc": {"iteraciones": 1}
        }
    )
    print(" U (Update): Experimento 'EXP-001' actualizado con éxito.")
    
    # Comprobación de lectura del documento actualizado
    doc_modificado = db.demo_crud.find_one({"codigo": "EXP-001"}, {"_id": 0})
    print("Documento resultante tras Update en db.demo_crud:")
    print(doc_modificado)
else:
    print("Omitido: Sin conexión a Atlas.")


 U (Update): Experimento 'EXP-001' actualizado con éxito.
Documento resultante tras Update en db.demo_crud:
{'codigo': 'EXP-001', 'modelo': 'random_forest', 'parametros': {'n_estimators': 150, 'max_depth': 15}, 'score_f1': 0.755, 'activo': True, 'iteraciones': 1, 'revisado': True}


#### Paso D (Delete): Eliminación con `db.demo_crud.delete_one()`


In [60]:
if db is not None:
    # D (Delete): Eliminamos el experimento inactivo 'EXP-003'
    del_res = db.demo_crud.delete_one({"codigo": "EXP-003"})
    print(f" D (Delete): {del_res.deleted_count} documento eliminado de db.demo_crud.")
    
    # Cantidad restante
    total_mongo = db.demo_crud.count_documents({})
    print(f"Documentos restantes en db.demo_crud: {total_mongo}")
else:
    print("Omitido: Sin conexión a Atlas.")


 D (Delete): 1 documento eliminado de db.demo_crud.
Documentos restantes en db.demo_crud: 3


---
### 3.3 Limpieza y Teardown Automatizado

Finalizada la demostración, destruimos la tabla `demo_crud` en SQLite y la colección `demo_crud` en MongoDB Atlas (`db.demo_crud.drop()`). Esto asegura que el entorno quede exactamente en el estado original sin dejar tablas ni datos residuales.


In [61]:
# 1. Teardown en SQLite: Eliminación física de la tabla de práctica
con_sql.execute("DROP TABLE IF EXISTS demo_crud;")
con_sql.commit()

# Verificación de que demo_crud ya no existe en el catálogo de SQLite
cursor_verif = con_sql.execute("""
SELECT name FROM sqlite_master WHERE type='table' AND name='demo_crud';
""").fetchone()

if cursor_verif is None:
    print(" Teardown SQLite: Tabla 'demo_crud' eliminada exitosamente. Integridad preservada.")
else:
    print(" Advertencia: La tabla 'demo_crud' aún existe en SQLite.")

# 2. Teardown en MongoDB Atlas: Eliminación de la colección de práctica
if db is not None:
    db.demo_crud.drop()
    if "demo_crud" not in db.list_collection_names():
        print(" Teardown MongoDB: Colección 'db.demo_crud' purgada exitosamente de Atlas.")
    else:
        print(" Advertencia: La colección 'demo_crud' aún persiste en Atlas.")
else:
    print("Teardown Mongo omitido (sin conexión).")

# Cierre ordenado de conexiones
con_sql.close()
print(" Demostración completada con éxito. Todos los recursos temporales han sido liberados.")


 Teardown SQLite: Tabla 'demo_crud' eliminada exitosamente. Integridad preservada.
 Teardown MongoDB: Colección 'db.demo_crud' purgada exitosamente de Atlas.
 Demostración completada con éxito. Todos los recursos temporales han sido liberados.
